# Data Types in Finance — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press

---
**Learning Objectives:**
- Distinguish time series, cross-sectional, and panel data
- Calculate simple and log returns
- Visualise and explore real financial data
- Interpret descriptive statistics in a finance context

> This notebook accompanies the lecture slides. Run each cell with **Shift+Enter**.


## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'; GREY = '#4B4B4B'
print('✓ Libraries loaded.')

---
# Part 1 — Time Series Data
**Definition:** Observations on the *same entity* recorded at *multiple points in time* — in sequential order.

Key property: **order matters**. Yesterday's price directly affects today's return.

### 1.1 Download Stock Price Data

In [ ]:
TICKER = 'AAPL'
START  = '2019-01-01'
END    = '2024-12-31'

data   = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
prices = data['Close'].squeeze()

print(f'Downloaded {TICKER}: {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'\nFirst 5 observations:')
print(prices.head().round(2))

### 1.2 Plot Price Series & Daily Returns

In [ ]:
ret = prices.pct_change().dropna() * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.05})
axes[0].plot(prices.index, prices.values, color='black', lw=1.3)
axes[0].fill_between(prices.index, prices.values, alpha=0.06, color='black')
axes[0].set_ylabel('Price (USD)')
axes[0].set_title(f'{TICKER} — Price & Daily Returns ({START[:4]}–{END[:4]})', fontweight='bold')

axes[1].bar(ret.index, ret.values,
            color=[RED if v < 0 else 'black' for v in ret.values], width=1.0)
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_ylabel('Daily return (%)')
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.show()
print('→ TIME SERIES: one entity, many time points, order matters')

---
# Part 2 — Simple vs. Log Returns

| | Simple Return | Log Return |
|--|--|--|
| **Formula** | $R_t = (P_t - P_{t-1})/P_{t-1}$ | $r_t = \ln(P_t/P_{t-1})$ |
| **Python** | `prices.pct_change()` | `np.log(prices/prices.shift(1))` |
| **Additive?** | No — must multiply | Yes — just sum |
| **Use for** | Reporting, portfolio returns | Modelling, statistics, regressions |

### 2.1 Calculate Both Return Types

In [ ]:
simple_ret = prices.pct_change().dropna()
log_ret    = np.log(prices / prices.shift(1)).dropna()

idx = simple_ret.index.intersection(log_ret.index)
s   = simple_ret.loc[idx] * 100
l   = log_ret.loc[idx]    * 100

print('Descriptive Statistics (%)')
print(pd.DataFrame({'Simple': s.describe(), 'Log': l.describe()}).round(5))
print(f'\nMax daily difference |simple − log|: {(s-l).abs().max():.5f}%')

### 2.2 Visualise: Daily, Cumulative, and Difference

In [ ]:
diff       = s - l
cum_simple = (1 + simple_ret).cumprod() - 1
cum_log    = log_ret.cumsum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# A: daily
axes[0].plot(s.index, s.values, color='black', lw=0.7, alpha=0.85, label='Simple')
axes[0].plot(l.index, l.values, color=ORANGE,  lw=0.7, alpha=0.85, ls='--', label='Log')
axes[0].axhline(0, color=GREY, lw=0.5)
axes[0].set_title('A: Daily — Almost Identical', fontweight='bold')
axes[0].set_ylabel('Return (%)')
axes[0].legend()

# B: cumulative
axes[1].plot(cum_simple.index, cum_simple*100, color='black', lw=1.8, label='Cumulative simple')
axes[1].plot(cum_log.index,    cum_log*100,    color=RED,     lw=1.8, ls='--', label='Sum of log')
axes[1].fill_between(cum_simple.index, cum_simple*100, cum_log*100, alpha=0.15, color=RED)
axes[1].set_title('B: Cumulative — Divergence Visible!', fontweight='bold')
axes[1].set_ylabel('Cumulative return (%)')
axes[1].legend()

# C: difference
axes[2].fill_between(diff.index, diff.values, where=diff.values>0,  color=RED,        alpha=0.55)
axes[2].fill_between(diff.index, diff.values, where=diff.values<=0, color='steelblue', alpha=0.55)
axes[2].axhline(0, color=GREY, lw=0.8)
axes[2].set_title('C: Difference (Simple − Log)', fontweight='bold')
axes[2].set_ylabel('Difference (%)')

plt.suptitle('Simple vs. Log Returns: Why It Matters', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key insights:')
print('• Daily: simple ≈ log (diff < 0.01%)')
print('• Cumulative: simple > log (diverge in bull markets)')
print('• Rule: log for modelling, simple for reporting')

### 2.3 Concrete Numerical Example — Round Trip

In [ ]:
print('=' * 60)
print('EXAMPLE: Crash ($100 → $70) and Recovery ($70 → $100)')
print('=' * 60)
P0, P1, P2 = 100, 70, 100

R1_s = (P1-P0)/P0;        R2_s = (P2-P1)/P1
R1_l = np.log(P1/P0);     R2_l = np.log(P2/P1)

print(f'\nDay 1: ${P0} → ${P1}')
print(f'  Simple: {R1_s*100:.2f}%   Log: {R1_l*100:.2f}%')
print(f'\nDay 2: ${P1} → ${P2}')
print(f'  Simple: {R2_s*100:.2f}%   Log: {R2_l*100:.2f}%')
print(f'\nTotal simple: {((1+R1_s)*(1+R2_s)-1)*100:.2f}%  (must multiply!)')
print(f'Total log:    {(R1_l+R2_l)*100:.2f}%  (just add!) ← back to 0')
print(f'\n→ Log returns are additive — key advantage for multi-period models')

---
# Part 3 — Cross-Sectional Data & Beta
**Definition:** Observations on *multiple entities* at a *single point in time*.

**Beta:** $\beta = \text{Cov}(r_i, r_m) / \text{Var}(r_m)$

### 3.1 Download Multiple Stocks & Estimate Beta

In [ ]:
tickers      = ['AAPL', 'MSFT', 'JPM', '^GSPC']
raw          = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)
prices_panel = raw['Close'].copy()
prices_panel.columns = ['AAPL', 'MSFT', 'JPM', 'SP500']
prices_panel = prices_panel.dropna()
ret_panel    = prices_panel.pct_change().dropna()

print(f'Panel shape: {ret_panel.shape}  ({ret_panel.shape[0]} days × {ret_panel.shape[1]} assets)')
print(f'\nSummary statistics (daily returns, %):')
print((ret_panel * 100).describe().round(4))

In [ ]:
results = {}
for col in ['AAPL', 'MSFT', 'JPM']:
    slope, intercept, r, p, se = stats.linregress(ret_panel['SP500'], ret_panel[col])
    results[col] = {'Beta': round(slope, 3),
                    'Alpha (daily %)': round(intercept*100, 4),
                    'R²': round(r**2, 3)}

print('Cross-Sectional Beta Estimates (2019–2024):')
print(pd.DataFrame(results).T)
print('\n→ CROSS-SECTIONAL: one beta per firm, one point in time')
print('→ Beta > 1: aggressive  |  Beta < 1: defensive')

---
# Part 4 — Panel Data
**Definition:** Multiple entities observed over multiple time periods.

### 4.1 Cumulative Returns & Monthly Heatmap

In [ ]:
!pip install seaborn --quiet
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cum = (1 + ret_panel).cumprod() - 1
for col, c in zip(cum.columns, ['black', ORANGE, '#0E75FE', RED]):
    axes[0].plot(cum.index, cum[col]*100, lw=1.6, color=c, label=col)
axes[0].axhline(0, color=GREY, lw=0.8)
axes[0].set_title('Cumulative Returns 2019–2024', fontweight='bold')
axes[0].set_ylabel('Cumulative return (%)')
axes[0].legend()

monthly = ret_panel.resample('ME').apply(lambda x: (1+x).prod()-1)
mp = monthly.iloc[-24:] * 100
mp.index = mp.index.strftime('%b %y')
sns.heatmap(mp.T, ax=axes[1], cmap='RdYlGn', center=0,
            annot=True, fmt='.0f', linewidths=0.5, linecolor='white',
            annot_kws={'size': 8}, cbar_kws={'label': 'Monthly return (%)'})
axes[1].set_title('Monthly Returns — Last 24 Months', fontweight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

---
# Part 5 — Annualised Statistics

In [ ]:
TRADING_DAYS = 252
RF = 0.04

print(f'Annualised Statistics (2019–2024) | Risk-free rate: {RF*100:.0f}%')
print('=' * 72)
for col in ret_panel.columns:
    r       = ret_panel[col]
    ann_ret = (1 + r.mean()) ** TRADING_DAYS - 1
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - RF) / ann_vol
    print(f'{col:<8} | Return: {ann_ret*100:6.1f}%  '
          f'| Vol: {ann_vol*100:5.1f}%  '
          f'| Sharpe: {sharpe:5.2f}')

print('\nFormulas:')
print('  Annual return = (1 + daily_mean)^252 − 1')
print('  Annual vol    = daily_std × √252')
print('  Sharpe        = (annual_return − risk_free) / annual_vol')

---
## Summary Table

| Concept | Key Formula | Python |
|---------|-------------|--------|
| Simple return | $R_t = (P_t-P_{t-1})/P_{t-1}$ | `prices.pct_change()` |
| Log return | $r_t = \ln(P_t/P_{t-1})$ | `np.log(prices/prices.shift(1))` |
| Annual vol | $\sigma_{ann} = \sigma_{daily} \times \sqrt{252}$ | `ret.std() * np.sqrt(252)` |
| Beta | $\beta = \text{Cov}(r_i, r_m)/\text{Var}(r_m)$ | `stats.linregress(market, stock)` |
| Sharpe | $(r - r_f) / \sigma$ | manual |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Statistical Properties of Financial Data (Fat Tails, Skewness, Volatility Clustering)*